In [1]:
pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00


In [2]:
!pip install transformers torch

In [3]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [4]:
import bitsandbytes

import torch
from transformers import pipeline

In [5]:
print("Downloading/Loading model... this takes a minute on the first run.")
pipe = pipeline(
    "text-generation",
    model="google/gemma-3-4b-it",
    device_map="auto",
    model_kwargs={
        "torch_dtype": torch.bfloat16,
        "quantization_config": {"load_in_4bit": True}
    }
)

Downloading/Loading model... this takes a minute on the first run.


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [8]:
def generate_qa(text_chunk, num_questions=5):
    # I slightly adjusted the JSON format to be a list of dictionaries
    # since you are asking for 5 pairs per chunk.
    prompt_text = f"""Based on the following book excerpt, generate {num_questions} expert-level Question-Answer pairs in JSON format.
Focus on deep logic and specific facts.
(your response is being recorded as a json data do not write anything extra than questions and answer DO NOT WRITE ANYTHING THAT CAN CORRUPT THE DATA)

TEXT: {text_chunk}

Follow the output format strictly. OUTPUT FORMAT:
[
  {{
    "question": "...",
    "answer": "..."
  }}
]"""

    # Local instruct models need data formatted as a conversation chat
    messages = [{"role": "user", "content": prompt_text}]

    # 2. Generate the response (Replaces model.generate_content)
    response = pipe(
        messages,
        max_new_tokens=1024,   # Gives the model enough room to write all {num_questions} pairs
        return_full_text=False # Crucial: ensures it ONLY returns the answer, not your prompt
    )

     # Extract just the text from the pipeline's output dictionary
    return response[0]['generated_text'].strip()

In [13]:
 #3. Process your book in chunks
print("Starting book processing...")
with open("/content/whale train.md", "r", encoding="utf-8") as f:
    full_text = f.read()

chunk_size = 4000 # You can adjust this value to change the chunk size
chunks = [full_text[i:i+chunk_size] for i in range(0, len(full_text), chunk_size)]

Starting book processing...


In [14]:
with open("/content/whale.json", "a", encoding="utf-8") as out:
    for i, chunk in enumerate(chunks):
        qa_data = generate_qa(chunk, num_questions=5)
        out.write(qa_data + "\n")
        print(f"Processed chunk {i+1}/{len(chunks)}...")

print("Finished generating Q&A pairs!")

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processed chunk 1/55...


Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processed chunk 2/55...


Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Processed chunk 52/55...


Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processed chunk 53/55...


Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processed chunk 54/55...
Processed chunk 55/55...
Finished generating Q&A pairs!
